# 06. End-to-End Recommendation Pipeline & Evaluation

Notebook này kết nối tất cả các Stage đơn lẻ lại thành một hệ thống gợi ý hoàn chỉnh 3 lớp (Retrieval -> Ranking -> Re-ranking) và tiến hành đánh giá toàn diện bằng các chỉ số Accuracy & Beyond-Accuracy.

---

### Cấu trúc luồng chạy thử nghiệm:
1.  **Stage 1: Retrieval**: Gọi cả hai mô hình **iALS** (Collaborative) và **TF-IDF Cosine** (Content-based) để lấy ra Top-150 candidates mỗi bên, gộp lại (Union) được khoảng ~250 candidates.
2.  **Stage 2: Ranking**: Dùng mô hình **LightGBM LGBMRanker** để chấm điểm chi tiết cho ~250 candidates của User.
3.  **Stage 3: Re-ranking**: Áp dụng **MMR (lambda=0.7)** để chọn ra Top-10 phim đa dạng và chất lượng nhất đưa tới client.
4.  **Evaluation**: Đánh giá dựa trên tập Test LOO bằng các chỉ số: **Hit Ratio@10 (HR@10)**, **NDCG@10**, **Diversity**, **Coverage**, **Novelty**.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import pickle

sys.path.append(os.path.abspath('..'))
from recsys_utils import (
    evaluate_implicit_loo, 
    calculate_beyond_accuracy_metrics, 
    BM25, 
    reciprocal_rank_fusion, 
    calculate_user_lambda
)

# Load models
with open("models/als_model.pkl", "rb") as f:
    als_model = pickle.load(f)
    
with open("models/bm25_model.pkl", "rb") as f:
    bm25 = pickle.load(f)
    
with open("models/lgb_ranker.pkl", "rb") as f:
    lgb_ranker = pickle.load(f)

# Load data
train_ratings = pd.read_csv("processed_data/train_ratings.csv")
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))
users_df = pd.read_csv(os.path.join("..", "..", "data", "simulator", "sim_users.csv"))

with open("processed_data/test_data.pkl", "rb") as f:
    test_data = pickle.load(f)

with open("processed_data/id_mappings.pkl", "rb") as f:
    user_to_idx, movie_to_idx, idx_to_movie = pickle.load(f)
    
with open("processed_data/user_item_matrix.pkl", "rb") as f:
    user_item_matrix = pickle.load(f)

# Tạo TF-IDF matrix cho MMR mở rộng
movies_df['genres'] = movies_df['genres'].fillna('')
movies_df['director'] = movies_df['director'].fillna('')
movies_df['cast'] = movies_df['cast'].fillna('')
movies_df['mmr_soup'] = movies_df.apply(
    lambda r: f"{r['genres'].replace('|', ' ')} {r['director'].replace(' ', '')} {' '.join(r['cast'].split('|')[:3])}", 
    axis=1
)
from sklearn.feature_extraction.text import TfidfVectorizer
mmr_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = mmr_vectorizer.fit_transform(movies_df['mmr_soup'])


In [ ]:
# 1. Định nghĩa End-to-End Pipeline
# Tạo metadata soup cho phim phục vụ tính cb_score của ứng viên
movies_df['director'] = movies_df['director'].fillna('')
movies_df['cast'] = movies_df['cast'].fillna('')
movies_df['keywords'] = movies_df['keywords'].fillna('')

def build_metadata_soup(row):
    genres = row['genres'].replace('|', ' ')
    cast = ' '.join(row['cast'].split('|')[:5])
    keywords = row['keywords'].replace('|', ' ')
    director = row['director'].replace(' ', '')
    return f"{genres} {director} {cast} {keywords}"

movies_df['soup'] = movies_df.apply(build_metadata_soup, axis=1)
movies_indexed_df = movies_df.set_index('movieId')
users_indexed_df = users_df.set_index('user_id')

# Danh sách tất cả các thể loại độc bản để tính entropy
all_genres = sorted(list(set([g for genres in movies_df['genres'].str.split('|').dropna() for g in genres if g])))

def end_to_end_recommend(user_id, top_k=10, custom_lambda=None):
    # --- STAGE 1: RETRIEVAL (BM25 + iALS -> RRF) ---
    liked_movies = train_ratings[train_ratings['userId'] == user_id]['movieId'].tolist()
    liked_set = set(liked_movies)
    
    # A. BM25 content candidate retrieval
    bm25_candidates = []
    user_bm25_all_scores = np.zeros(len(movies_indexed_df))
    liked_soups = [movies_indexed_df.loc[lid, 'soup'] for lid in liked_movies if lid in movies_indexed_df.index]
    if liked_soups:
        query = " ".join(liked_soups)
        user_bm25_all_scores = bm25.transform(query)
        sorted_cb_idx = np.argsort(user_bm25_all_scores)[::-1]
        
        # Top 100 ứng viên BM25 (chưa xem)
        for idx in sorted_cb_idx:
            mid = movies_df.iloc[idx]['movieId']
            if mid not in liked_set:
                bm25_candidates.append(mid)
            if len(bm25_candidates) >= 100:
                break
                
    # B. iALS collaborative candidate retrieval
    u_idx = user_to_idx.get(user_id, None)
    als_candidates = []
    if u_idx is not None:
        ids, _ = als_model.recommend(u_idx, user_item_matrix[u_idx], N=100)
        als_candidates = [idx_to_movie[i] for i in ids if i in idx_to_movie and idx_to_movie[i] not in liked_set]
        
    # C. Hợp nhất bằng RRF
    rrf_list = reciprocal_rank_fusion(als_candidates, bm25_candidates, k=60)
    candidates = [item[0] for item in rrf_list[:250]]
    
    if not candidates:
        candidates = movies_df.sort_values(by='popularity', ascending=False)['movieId'].head(100).tolist()
        candidates = [cid for cid in candidates if cid not in liked_set]
        
    # --- STAGE 2: RANKING (LightGBM) ---
    features = []
    valid_candidates = []
    
    als_user_factors = als_model.user_factors
    als_item_factors = als_model.item_factors
    
    for mid in candidates:
        if mid not in movies_indexed_df.index:
            continue
        valid_candidates.append(mid)
        movie = movies_indexed_df.loc[mid]
        user = users_indexed_df.loc[user_id]
        
        favorite_genres = set(user['favorite_genres'].split('|'))
        movie_genres = set(movie['genres'].split('|'))
        genre_overlap = len(favorite_genres.intersection(movie_genres))
        
        try:
            release_year = int(str(movie['release_date'])[:4])
        except:
            release_year = 2010
            
        u_idx = user_to_idx.get(user_id, None)
        m_idx = movie_to_idx.get(mid, None)
        
        als_score = als_user_factors[u_idx].dot(als_item_factors[m_idx]) if (u_idx is not None and m_idx is not None) else 0.0
        cb_score = user_bm25_all_scores[m_idx] if (m_idx is not None) else 0.0
            
        features.append({
            'popularity': movie['popularity'],
            'vote_average': movie['vote_average'],
            'genre_overlap': genre_overlap,
            'release_year': release_year,
            'user_activity': user['activity_level'],
            'user_bias': user['user_bias'],
            'als_score': als_score,
            'cb_score': cb_score
        })
        
    X_pred = pd.DataFrame(features)
    scores = lgb_ranker.predict(X_pred)
    
    candidate_scores = list(zip(valid_candidates, scores))
    candidate_scores.sort(key=lambda x: x[1], reverse=True)
    
    # --- STAGE 3: RE-RANKING (MMR với Lambda động) ---
    # Tính lambda động nếu không chỉ định cụ thể
    if custom_lambda is not None:
        lambda_val = custom_lambda
    else:
        # Lấy lịch sử thể loại phim đã xem ở tập train
        user_history_mids = train_ratings[train_ratings['userId'] == user_id]['movieId'].tolist()
        user_history_genres = []
        for hmid in user_history_mids:
            if hmid in movies_indexed_df.index:
                user_history_genres.extend(movies_indexed_df.loc[hmid, 'genres'].split('|'))
        lambda_val = calculate_user_lambda(user_history_genres, all_genres, base_min=0.4, base_max=0.9)
        
    from sklearn.metrics.pairwise import cosine_similarity
    
    final_recs = []
    if candidate_scores:
        candidates_ids = [item[0] for item in candidate_scores]
        scores_arr = np.array([item[1] for item in candidate_scores])
        
        if scores_arr.max() != scores_arr.min():
            scores_norm = (scores_arr - scores_arr.min()) / (scores_arr.max() - scores_arr.min())
        else:
            scores_norm = np.ones_like(scores_arr)
            
        selected_items = []
        unselected_indices = list(range(len(candidates_ids)))
        
        first_choice = np.argmax(scores_norm)
        selected_items.append(candidates_ids[first_choice])
        unselected_indices.remove(first_choice)
        
        while len(selected_items) < top_k and unselected_indices:
            selected_matrix_indices = [movie_to_idx[mid] for mid in selected_items if mid in movie_to_idx]
            if not selected_matrix_indices:
                break
            selected_vectors = tfidf_matrix[selected_matrix_indices]
            
            valid_unselected = []
            valid_matrix_indices = []
            for idx in unselected_indices:
                cid = candidates_ids[idx]
                m_idx = movie_to_idx.get(cid, None)
                if m_idx is not None:
                    valid_unselected.append(idx)
                    valid_matrix_indices.append(m_idx)
            
            if not valid_unselected:
                break
                
            unselected_vectors = tfidf_matrix[valid_matrix_indices]
            sim_matrix = cosine_similarity(unselected_vectors, selected_vectors)
            max_sim = sim_matrix.max(axis=1)
            
            best_mmr = -1e9
            best_idx_in_unselected = -1
            
            for i, idx in enumerate(valid_unselected):
                mmr_val = lambda_val * scores_norm[idx] - (1 - lambda_val) * max_sim[i]
                if mmr_val > best_mmr:
                    best_mmr = mmr_val
                    best_idx_in_unselected = idx
                    
            if best_idx_in_unselected == -1:
                break
            selected_items.append(candidates_ids[best_idx_in_unselected])
            unselected_indices.remove(best_idx_in_unselected)
        final_recs = selected_items
        
    return final_recs

print("Pipeline End-to-End đã xây dựng xong!")


In [ ]:
# 2. Đánh giá thử nghiệm hệ thống
predictions_dict = {}
pipeline_recs = {}

print("Bắt đầu đánh giá Pipeline trên 200 users từ tập test LOO (RRF + BM25 + Dynamic Lambda)...")
for u, pos_item, neg_items in test_data:
    u = int(u)
    pos_item = int(pos_item)
    neg_items = [int(x) for x in neg_items]
    
    recs = end_to_end_recommend(u, top_k=10, custom_lambda=None)
    pipeline_recs[u] = recs
    
    items = [pos_item] + neg_items
    
    user_preds = []
    for item in items:
        if item in recs:
            score = 10 - recs.index(item)
        else:
            score = 0
        user_preds.append((item, score, item == pos_item))
    predictions_dict[u] = user_preds

# 3. Tính toán các metrics
hr, ndcg, mrr = evaluate_implicit_loo(predictions_dict, k=10)
div, nov, cov = calculate_beyond_accuracy_metrics(
    pipeline_recs, train_ratings, movies_df, movie_features=None, k=10, item_col='movieId'
)

print("\n=== KẾT QUẢ ĐÁNH GIÁ END-TO-END PIPELINE (THUẦN ML) ===")
print(f"Hit Ratio@10 (HR@10):  {hr:.4f}")
print(f"NDCG@10:               {ndcg:.4f}")
print(f"Mean Reciprocal Rank:  {mrr:.4f}")
print(f"Diversity@10:          {div:.4f}")
print(f"Novelty@10:            {nov:.4f}")
print(f"Coverage@10:           {cov:.4f}")


## Kết luận và Giải pháp đề xuất cho dự án

*   **Hiệu năng vượt trội**: Pipeline thuần ML kết hợp Stage 1 (iALS + CB) -> Stage 2 (LightGBM) -> Stage 3 (MMR) mang lại kết quả chất lượng vượt trội nhờ khả năng tối ưu hóa đa lớp.
*   **Cold Start được xử lý**:
    *   Nhờ nhánh **Content-Based TF-IDF** ở Stage 1, các phim mới 2026 hoàn toàn có thể được chọn làm ứng viên và đưa vào Ranker để gợi ý ngay lập tức.
*   **Khả năng phân tách & diễn giải (Interpretability)**:
    *   LightGBM cho phép phân tích Feature Importance để giải thích lý do xếp hạng.
    *   MMR kiểm soát trực tiếp độ đa dạng của danh sách phim để đáp ứng thị huớng phong phú của người dùng.
